<a href="https://colab.research.google.com/github/saleet-developer/Medical-RAG-Chatbot/blob/main/notebooks/colab/data-preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
project_path = '/content/drive/MyDrive/Medical-RAG-Project'
os.listdir(project_path)

['raw data',
 'processed data',
 'models',
 'checkpoints',
 'outputs',
 'embedding.ipynb',
 'preprocess.ipynb']

In [4]:
PROJECT_PATH = '/content/drive/MyDrive/Medical-RAG-Project'
RAW_PATH = '/content/drive/MyDrive/Medical-RAG-Project/raw data'
PROCESSED_PATH = '/content/drive/MyDrive/Medical-RAG-Project/processed data'

print(f'Raw Path {RAW_PATH}')
print(f'Processed Path {PROCESSED_PATH}')

Raw Path /content/drive/MyDrive/Medical-RAG-Project/raw data
Processed Path /content/drive/MyDrive/Medical-RAG-Project/processed data


In [5]:
# !pip install datasets transformers tqdm

In [6]:
import zipfile
import os
# file_path = os.path.join(RAW_PATH, 'PubMed-Abstractions', 'PubMed.zip')

# with zipfile.ZipFile(file_path, 'r') as zip_ref:
#   zip_ref.extractall(RAW_PATH)

In [7]:
def parse_pubmed_rct(filepath):
    abstracts = []
    current_pmid = None
    current_sentences = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            clean_line = line.strip()  # This handles \r, \n, and spaces

            # If the line is truly empty after stripping
            if not clean_line:
                if current_pmid and current_sentences:
                    abstracts.append({
                        'pmid': current_pmid,
                        'text': ' '.join(current_sentences),
                        'sentences': current_sentences.copy()
                    })
                    current_pmid = None
                    current_sentences = []
                continue

            if clean_line.startswith('###'):
                current_pmid = clean_line[3:].strip()
                current_sentences = []
            else:
                # Use the original line for splitting to preserve tabs
                if '\t' in line:
                    _, sentence = line.rstrip('\n').split('\t', 1)
                    current_sentences.append(sentence)
                else:
                    current_sentences.append(clean_line)

        # Final catch for the last abstract in the file
        if current_pmid and current_sentences:
            abstracts.append({
                'pmid': current_pmid,
                'text': ' '.join(current_sentences),
                'sentences': current_sentences.copy()
            })

    return abstracts

In [8]:
train_file = '/content/drive/MyDrive/Medical-RAG-Project/raw data/PubMed_20k_RCT/train.txt'

print("File exists:", os.path.exists(train_file))
print("File size (bytes):", os.path.getsize(train_file))

with open(train_file, 'r', encoding='utf-8') as f:
    sample = f.read(500)
print("\nFirst 500 characters:\n", sample)



File exists: True
File size (bytes): 29384101

First 500 characters:
 ###24293578
OBJECTIVE	To investigate the efficacy of 6 weeks of daily low-dose oral prednisolone in improving pain , mobility , and systemic low-grade inflammation in the short term and whether the effect would be sustained at 12 weeks in older adults with moderate to severe knee osteoarthritis ( OA ) .
METHODS	A total of 125 patients with primary knee OA were randomized 1:1 ; 63 received 7.5 mg/day of prednisolone and 62 received placebo for 6 weeks .
METHODS	Outcome measures included pain redu


In [9]:
abstracts = parse_pubmed_rct(train_file)
print(f"Extracted {len(abstracts)} abstracts")

Extracted 15000 abstracts


In [10]:
import re

target_diseases = ['diabetes', 'ophthalmology', 'cardiology',
                   'hypertension', 'heart', 'osteoarthritis',
                   'arthritis', 'obesity', 'cancer', 'stroke']

def contains_disease(text, diseases):
    text_lower = text.lower()
    return any(disease in text_lower for disease in diseases)

filtered_abstracts = [a for a in abstracts if contains_disease(a['text'], target_diseases)]
print(f"Filtered to {len(filtered_abstracts)} abstracts relevant to your domains")

Filtered to 4228 abstracts relevant to your domains


In [11]:
def extract_section_metadata(abstract):
    """Count sentences by section type"""
    # This requires re-parsing the original file with labels
    # For now, we'll just note that we have full abstracts
    return {
        'source': 'pubmed_rct',
        'has_sentence_labels': True,
        'pmid': abstract.get('pmid', '')
    }

# Add metadata to each abstract
for i, abstract in enumerate(filtered_abstracts):
    abstract['metadata'] = extract_section_metadata(abstract)

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunked_data = []
for idx, abstract in enumerate(filtered_abstracts):
    text = abstract['text']
    chunks = text_splitter.split_text(text)

    for chunk_idx, chunk in enumerate(chunks):
        chunked_data.append({
            'doc_id': f"pubmed_{idx}_{chunk_idx}",
            'text': chunk,
            'source': 'pubmed_rct',
            'pmid': abstract.get('pmid', ''),
            'chunk_index': chunk_idx,
            'has_labels': True
        })

print(f"Created {len(chunked_data)} chunks")

Created 20917 chunks


In [16]:
outfile = '/content/drive/MyDrive/Medical-RAG-Project/processed data/Chunked-Documents/pubmed_rct_chunks.jsonl'
with open(outfile, 'w') as f:
  for item in chunked_data:
      f.write(json.dumps(item) + '\n')

print(f'Saved {len(chunked_data)} chunks')


Saved 20917 chunks


In [17]:
file_path = os.path.join(RAW_PATH, 'MedQuAD', 'MedQuAD.zip')

with zipfile.ZipFile(file_path, 'r') as zip_ref:
  zip_ref.extractall(RAW_PATH)

In [18]:
import pandas as pd
medquad_file = os.path.join(RAW_PATH, 'medquad.csv')
if os.path.exists(medquad_file):
   df_med = pd.read_csv(medquad_file)
   print(f'Loaded {len(df_med)} Q&A pairs')
else:
  print(f'file not found')
  df_med = pd.DataFrame()

Loaded 16412 Q&A pairs


In [19]:
print(df_med.columns)
print(df_med.head())

Index(['question', 'answer', 'source', 'focus_area'], dtype='object')
                                 question  \
0                What is (are) Glaucoma ?   
1                  What causes Glaucoma ?   
2     What are the symptoms of Glaucoma ?   
3  What are the treatments for Glaucoma ?   
4                What is (are) Glaucoma ?   

                                              answer           source  \
0  Glaucoma is a group of diseases that can damag...  NIHSeniorHealth   
1  Nearly 2.7 million people have glaucoma, a lea...  NIHSeniorHealth   
2  Symptoms of Glaucoma  Glaucoma can develop in ...  NIHSeniorHealth   
3  Although open-angle glaucoma cannot be cured, ...  NIHSeniorHealth   
4  Glaucoma is a group of diseases that can damag...  NIHSeniorHealth   

  focus_area  
0   Glaucoma  
1   Glaucoma  
2   Glaucoma  
3   Glaucoma  
4   Glaucoma  


In [23]:
def contain_disease(text):
    text_lower = str(text).lower()
    return any(disease in text_lower for disease in target_diseases)

df_med_filtered = df_med[
    df_med['question'].apply(contain_disease) | df_med['answer'].apply(contain_disease)
].reset_index(drop=True)

print(f'Filered to {len(df_med_filtered)} Q&A pairs')

Filered to 4737 Q&A pairs


In [24]:
df_med_filtered['combined'] = 'Questions' + df_med_filtered['question'] + '\nAnswer:' + df_med_filtered['answer']

In [26]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)

med_chunks = []
for idx, row in df_med_filtered.iterrows():
    text = row['combined']
    chunks = text_splitter.split_text(text)
    for cidx, chunk in enumerate(chunks):
        med_chunks.append({
            'doc_id': f"medquad_{idx}_{cidx}",
            'text': chunk,
            'source': 'medquad',
            'disease': row.get('disease', 'unknown'),
            'chunk_index': cidx
        })

print(f"Created {len(med_chunks)} chunks from MedQuAD")

Created 31862 chunks from MedQuAD


In [42]:
outfile = '/content/drive/MyDrive/Medical-RAG-Project/processed data/Chunked-Documents/medquad_chunks.jsonl'
with open(outfile, 'w') as f:
  for item in med_chunks:
      f.write(json.dumps(item) + '\n')

print(f'Saved {len(med_chunks)} chunks')

Saved 31862 chunks


In [29]:
file_path = os.path.join(RAW_PATH, 'HealthCareMegic', 'HealthCareMagic.zip')

with zipfile.ZipFile(file_path, 'r') as zip_ref:
  zip_ref.extractall(RAW_PATH)

In [31]:
with open(os.path.join(RAW_PATH, 'HealthCareMagic-100k.json'), 'r') as f:
    data = json.load(f)  # data is a list of dicts
df_hm = pd.DataFrame(data)
print(df_hm.head())

                                         instruction  \
0  If you are a doctor, please answer the medical...   
1  If you are a doctor, please answer the medical...   
2  If you are a doctor, please answer the medical...   
3  If you are a doctor, please answer the medical...   
4  If you are a doctor, please answer the medical...   

                                               input  \
0  I woke up this morning feeling the whole room ...   
1  My baby has been pooing 5-6 times a day for a ...   
2  Hello, My husband is taking Oxycodone due to a...   
3  lump under left nipple and stomach pain (male)...   
4  I have a 5 month old baby who is very congeste...   

                                              output  
0  Hi, Thank you for posting your query. The most...  
1  Hi... Thank you for consulting in Chat Doctor....  
2  Hello, and I hope I can help you today.First, ...  
3  HI. You have two different problems. The lump ...  
4  Thank you for using Chat Doctor. I would sugge..

In [33]:
df_hm = df_hm.rename(columns={'input': 'question', 'output': 'answer'})
df_hm = df_hm.drop(columns=['instruction'], errors='ignore')

In [36]:
mask = df_hm['question'].apply(contain_disease) | df_hm['answer'].apply(contain_disease)
df_hm_filtered = df_hm[mask].reset_index(drop=True)
print(f"Filtered to {len(df_hm_filtered)} Q&A pairs")

Filtered to 23870 Q&A pairs


In [37]:
df_hm_filtered['combined'] = "Question: " + df_hm_filtered['question'] + "\nAnswer: " + df_hm_filtered['answer']

In [38]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)

hm_chunks = []
for idx, row in df_hm_filtered.iterrows():
    chunks = text_splitter.split_text(row['combined'])
    for cidx, chunk in enumerate(chunks):
        hm_chunks.append({
            'doc_id': f"hm_{idx}_{cidx}",
            'text': chunk,
            'source': 'healthcaremagic',
            'disease': row.get('disease', 'unknown'),
            'chunk_index': cidx
        })

print(f"Created {len(hm_chunks)} chunks from HealthCareMagic")


Created 79615 chunks from HealthCareMagic


In [41]:
outfile = '/content/drive/MyDrive/Medical-RAG-Project/processed data/Chunked-Documents/healthcaremagic_chunks.jsonl'
with open(outfile, 'w') as f:
  for item in hm_chunks:
      f.write(json.dumps(item) + '\n')

print(f'Saved {len(hm_chunks)} chunks')

Saved 79615 chunks


In [44]:
all_chunks = []

pubmed_json = os.path.join(PROCESSED_PATH, 'Chunked-Documents', 'pubmed_rct_chunks.jsonl')
if os.path.exists(pubmed_json):
   with open(pubmed_json) as f:
        for line in f:
            all_chunks.append(json.loads(line))
   print(f'Loaded {len([c for c in all_chunks if c['source']=='pubmed_rct'])} PubMed Chunks')
else:
  print(f'pubmed json not found')

medquad_json = os.path.join(PROCESSED_PATH, 'Chunked-Documents', 'medquad_chunks.jsonl')
if os.path.exists(medquad_json):
   with open(medquad_json) as f:
        for line in f:
            all_chunks.append(json.loads(line))
   print(f'Loaded {len([c for c in all_chunks if c['source']=='medquad'])} PubMed Chunks')
else:
  print(f'medquad json not found')


hm_json = os.path.join(PROCESSED_PATH, 'Chunked-Documents', 'healthcaremagic_chunks.jsonl')
if os.path.exists(hm_json):
   with open(hm_json) as f:
        for line in f:
            all_chunks.append(json.loads(line))
   print(f'Loaded {len([c for c in all_chunks if c['source']=='healthcaremagic'])} PubMed Chunks')
else:
  print(f'healthcaremagic json not found')


print(f"Total chunks combined: {len(all_chunks)}")


Loaded 20917 PubMed Chunks
Loaded 31862 PubMed Chunks
Loaded 79615 PubMed Chunks
Total chunks combined: 132394


In [45]:
combined_file = os.path.join(PROCESSED_PATH, 'Chunked-Documents', 'all_chunks.jsonl')

with open(combined_file, 'w') as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk) + '\n')

print(f"Saved combined file: {combined_file}")


Saved combined file: /content/drive/MyDrive/Medical-RAG-Project/processed data/Chunked-Documents/all_chunks.jsonl
